In [43]:
import importlib.util
if importlib.util.find_spec("spytial") is None:
    try:
        import piplite
        await piplite.install("spytial-diagramming")
    except ImportError:
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "spytial-diagramming"])
from spytial import *
import spytial.utils as _su
if not _su.is_notebook():
    _su.is_notebook = lambda: True  # JupyterLite/Pyodide: render diagram() inline
from spytial.annotations import *

# Max Heap

In [44]:
from typing import List

HEAP_VALUES = "((int - 0).(list.idx))"
HEAP_VALUE_TO_INDEX = f"({HEAP_VALUES} <: (~(list.idx)))"
LEFT_CHILDREN = "(~(list.idx)).({ i, j : (int-0) | j = multiply[i,2] }).(list.idx)" #"{ parent, child : ((int - 0).(list.idx)) | (some i : int | (i->parent + (multiply[@num:i, 2])->child) in (list.idx) )}"
RIGHT_CHILDREN = "(~(list.idx)).({ i, j : (int-0) | j = add[1, multiply[i, 2]] }).(list.idx)"   #"{ parent, child : ((int - 0).(list.idx)) | (some i : int | (i->parent + (add[1, multiply[@num:i, 2]])->child) in (list.idx) )}"
@hideAtom(selector=f'MaxHeap + list + (int - {HEAP_VALUES})')
@orientation(selector= LEFT_CHILDREN, directions=["left", "below"])
@inferredEdge(selector=LEFT_CHILDREN, name = "left")
@orientation(selector=RIGHT_CHILDREN, directions=["right", "below"])
@inferredEdge(selector=RIGHT_CHILDREN, name = "right")
@tag(toTag=HEAP_VALUES, name="index", value=HEAP_VALUE_TO_INDEX)
class MaxHeap:
    """
    CLRS-style max heap storing integers.
    1-indexed: a[0] unused.
    """
    def __init__(self, data: List[int] = None):
        self.a: List[int] = [0]
        if data:
            self.a.extend(data)
        self.n = len(self.a) - 1
        if self.n > 1:
            self.build_max_heap()

    # index helpers
    @staticmethod
    def _parent(i: int) -> int: return i // 2
    @staticmethod
    def _left(i: int) -> int:   return 2 * i
    @staticmethod
    def _right(i: int) -> int:  return 2 * i + 1

    def _max_heapify(self, i: int) -> None:
        while True:
            l, r = self._left(i), self._right(i)
            largest = i
            if l <= self.n and self.a[l] > self.a[largest]:
                largest = l
            if r <= self.n and self.a[r] > self.a[largest]:
                largest = r
            if largest == i:
                break
            self.a[i], self.a[largest] = self.a[largest], self.a[i]
            i = largest

    def build_max_heap(self) -> None:
        for i in range(self.n // 2, 0, -1):
            self._max_heapify(i)

    # API
    def max(self) -> int:
        if self.n < 1:
            raise IndexError("heap underflow")
        return self.a[1]

    def extract_max(self) -> int:
        if self.n < 1:
            raise IndexError("heap underflow")
        m = self.a[1]
        self.a[1] = self.a[self.n]
        self.a.pop()
        self.n -= 1
        if self.n >= 1:
            self._max_heapify(1)
        return m

    def increase_key(self, i: int, key: int) -> None:
        if i < 1 or i > self.n:
            raise IndexError("index out of range")
        if key < self.a[i]:
            raise ValueError("new key is smaller than current key")
        self.a[i] = key
        while i > 1 and self.a[self._parent(i)] < self.a[i]:
            p = self._parent(i)
            self.a[i], self.a[p] = self.a[p], self.a[i]
            i = p

    def insert(self, key: int) -> None:
        self.n += 1
        self.a.append(float("-inf"))  # sentinel
        self.increase_key(self.n, key)

    def __len__(self) -> int:
        return self.n

    def __repr__(self) -> str:
        return f"MaxHeap({self.a[1:]})"



## Max Heap In CLRS

![max-heap](./img/max-heap.png)

In [45]:
h = MaxHeap([16, 14, 10, 8, 7, 9, 3, 2, 4, 1])
diagram(h)


# Fibonacci Heap



In [46]:
import math


# Fibonacci Heap (priority queue) with spatial annotations for visualization
PARENT_CHILD = "(~parent) & (FibNode -> FibNode)"
SAME_DEPTH = "{x, y : FibNode | #(x.^parent & FibNode) = #(y.^parent & FibNode)} - iden"
LINEAR_RIGHT = "(right & (FibNode -> FibNode)) - (FibNode -> (FibonacciHeap.min + FibNode.child))"
SUBTREE = "(iden + ^(~parent)) & (FibNode -> FibNode)"
ORDERED_SUBTREES = f"(~({SUBTREE})).({LINEAR_RIGHT}).({SUBTREE})"
MARKED = "{x : FibNode | @bool:(x.mark)}"
UNMARKED = f"FibNode - ({MARKED})"

@attribute(field="key", selector=UNMARKED, textStyle=TextStyle(size="large"))
@attribute(field="key", selector=MARKED, textStyle=TextStyle(size="large", color="white"))
@atomStyle(
    selector=MARKED,
    fillStyle=FillStyle(color="black"),
    borderStyle=BorderStyle(color="black", width=2),
    textStyle=TextStyle(color="white"),
)
#@hideField(field="parent")
@hideField(field="child")
@hideField(field="left")
@hideField(field="right")
#@inferredEdge(selector=PARENT_CHILD, name="child")
@orientation(selector=PARENT_CHILD, directions=["below"])
@align(selector=SAME_DEPTH, direction="horizontal")
@inferredEdge(
    selector=f"({LINEAR_RIGHT}) & ((FibNode - parent.FibNode) -> (FibNode - parent.FibNode))",
    name="",
    lineStyle=LineStyle(pattern="dashed"),
)
@orientation(selector=ORDERED_SUBTREES, directions=["right"])
class FibNode:
    def __init__(self, key=None):
        self.key = key
        self.degree = 0
        self.mark = False
        self.parent = None
        self.child = None
        # circular doubly-linked list pointers
        self.left = self
        self.right = self


@hideAtom(selector="NoneType + int + bool")
@hideField(field="n")
@orientation(selector="min", directions=["directlyBelow"])
class FibonacciHeap:
    def __init__(self):
        self.min: FibNode | None = None
        self.n: int = 0

    # helper: iterate a circular doubly-linked list starting at node
    def _iterate(self, start: FibNode):
        if start is None:
            return
        node = start
        while True:
            yield node
            node = node.right
            if node is start:
                break

    # insert a new key, return the node
    def insert(self, key):
        x = FibNode(key=key)
        # add to root list
        if self.min is None:
            self.min = x
        else:
            # insert x to the right of min
            x.right = self.min.right
            x.left = self.min
            self.min.right.left = x
            self.min.right = x
            if x.key < self.min.key:
                self.min = x
        self.n += 1
        return x

    def find_min(self):
        return self.min

    # merge another heap into this one (destructive)
    def union(self, other: "FibonacciHeap"):
        if other is None or other.min is None:
            return
        if self.min is None:
            self.min = other.min
            self.n = other.n
            return
        # concatenate root lists
        a = self.min.right
        b = other.min.left
        self.min.right = other.min
        other.min.left = self.min
        a.left = b
        b.right = a
        if other.min.key < self.min.key:
            self.min = other.min
        self.n += other.n

    # extract the minimum node and return it
    def extract_min(self):
        z = self.min
        if z is not None:
            # move z's children to root list
            if z.child is not None:
                children = list(self._iterate(z.child))
                for x in children:
                    # remove parent link
                    x.parent = None
                    # splice x into root list (to the right of min)
                    x.left = self.min
                    x.right = self.min.right
                    self.min.right.left = x
                    self.min.right = x
            # remove z from root list
            if z is z.right:
                self.min = None
            else:
                z.left.right = z.right
                z.right.left = z.left
                self.min = z.right
                self._consolidate()
            self.n -= 1
        return z

    def _consolidate(self):
        if self.min is None:
            return
        # upper bound on degree
        max_degree = int(math.log(self.n, 2)) + 2 if self.n > 0 else 1
        A = [None] * (max_degree + 1)
        roots = list(self._iterate(self.min))
        for w in roots:
            x = w
            d = x.degree
            while A[d] is not None:
                y = A[d]
                if x.key > y.key:
                    x, y = y, x
                self._link(y, x)
                A[d] = None
                d += 1
            A[d] = x
        # rebuild root list and find new min
        self.min = None
        for node in A:
            if node is not None:
                # isolate node
                node.left = node.right = node
                if self.min is None:
                    self.min = node
                else:
                    # insert to root list
                    node.right = self.min.right
                    node.left = self.min
                    self.min.right.left = node
                    self.min.right = node
                    if node.key < self.min.key:
                        self.min = node

    def _link(self, y: FibNode, x: FibNode):
        # remove y from root list
        y.left.right = y.right
        y.right.left = y.left
        # make y a child of x
        y.parent = x
        y.left = y.right = y
        if x.child is None:
            x.child = y
        else:
            # insert y into x's child circular list
            y.right = x.child.right
            y.left = x.child
            x.child.right.left = y
            x.child.right = y
        x.degree += 1
        y.mark = False

    def decrease_key(self, x: FibNode, k):
        if k > x.key:
            raise ValueError("new key is greater than current key")
        x.key = k
        y = x.parent
        if y is not None and x.key < y.key:
            self._cut(x, y)
            self._cascading_cut(y)
        if x.key < self.min.key:
            self.min = x

    def _cut(self, x: FibNode, y: FibNode):
        # remove x from y's child list
        if y.child is x:
            # if x is the only child
            if x.right is x:
                y.child = None
            else:
                y.child = x.right
        x.left.right = x.right
        x.right.left = x.left
        y.degree -= 1
        # add x to root list
        x.left = self.min
        x.right = self.min.right
        self.min.right.left = x
        self.min.right = x
        x.parent = None
        x.mark = False

    def _cascading_cut(self, y: FibNode):
        z = y.parent
        if z is not None:
            if not y.mark:
                y.mark = True
            else:
                self._cut(y, z)
                self._cascading_cut(z)

    def delete(self, x: FibNode):
        # decrease key to -infinity then extract min
        self.decrease_key(x, -float("inf"))
        self.extract_min()


![fh.png](./img/fibonacci-heap.png)

In [47]:
# Construct the 14-node Fibonacci heap from CLRS Figure 19.2(a).
H = FibonacciHeap()
nodes = {}

# This insertion order produces the circular root order 23, 7, 3, 17, 24.
for key in (23, 7, 3, 24, 17):
    nodes[key] = H.insert(key)

# Insert the remaining nodes as roots before linking them into the trees.
for key in (18, 52, 38, 39, 41, 30, 26, 46, 35):
    nodes[key] = H.insert(key)

# Build the one-child subtrees first.
H._link(nodes[39], nodes[18])
H._link(nodes[41], nodes[38])
H._link(nodes[30], nodes[17])
H._link(nodes[35], nodes[26])

# _link inserts after parent.child, so this order produces 18, 52, 38.
H._link(nodes[18], nodes[3])
H._link(nodes[38], nodes[3])
H._link(nodes[52], nodes[3])

# Children of 24 appear as 26, 46.
H._link(nodes[26], nodes[24])
H._link(nodes[46], nodes[24])

# Black nodes in Figure 19.2(a).
for key in (18, 39, 26):
    nodes[key].mark = True

# AND just to be 


diagram(H)